# jax-compress

### Description

This project started as a JAX/Flax port of [tensorflow-compress](https://github.com/byronknoll/tensorflow-compress).

jax-compress performs lossless data compression using neural networks. It can run on TPUs/GPUs with a large batch size to get a substantial speed improvement. It is made using Colab, which should make it easy to run through a web browser. You can choose a file, perform compression (or decompression), and download the result.

The neural network is trained from scratch during compression and decompression, so the model weights do not need to be stored. Arithmetic coding is used to encode the model predictions to a file.

Feel free to contact me at byron@byronknoll.com if you have any questions.

### Instructions:

Basic usage: configure all the fields in the "Parameters" section and select Runtime->Run All.

Advanced usage: save a copy of this notebook and modify the code.

### Related Projects
*   [tensorflow-compress](https://github.com/byronknoll/tensorflow-compress)
*   [NNCP](https://bellard.org/nncp/)
*   [lstm-compress](https://github.com/byronknoll/lstm-compress)
*   [cmix](http://www.byronknoll.com/cmix.html)

### Benchmarks
These benchmarks were performed using jax-compress v1 with the default parameter settings. v6e-1 TPU was used. Compression time and decompression time are approximately the same.
*   enwik8: compressed to 15,458,356 bytes in 13,699.63 seconds.
*   enwik9: compressed to 113,393,442 bytes in 110,013.19 seconds. Dictionary size: 80,040 bytes. The preprocessed enwik9 file was split into two parts. The "checkpoint" option was used to save/load model weights between processing each part. Article ordering preprocessing was used:
```
cd enwik9-preproc/
make
./enwik9-preproc c enwik9
# After decompression:
./enwik9-preproc d final.dat
```

See the [Large Text Compression Benchmark](http://mattmahoney.net/dc/text.html) for more information about the test files and a comparison with other programs.

### Versions
* v1 - released March 15, 2026. Changes from tensorflow-compress:
  * Added "retraining": similar to NNCP, the model gets periodically retrained using previously processed data
  * Fixed a bug in NNCP preprocessor

## Parameters

In [ ]:
#@markdown ---
batch_size = 128 #@param {type:"integer"}
#@markdown _This will split the file into N batches, and process them in parallel. Increasing this will improve speed but can make compression rate worse. Make this a multiple of 8 to improve speed on certain GPUs._

#@markdown ---
seq_length =  15 #@param {type:"integer"}
#@markdown _This determines the horizon for back propagation through time. Reducing this will improve speed, but can make compression rate worse._

#@markdown ---
rnn_units =  1400 #@param {type:"integer"}
#@markdown _This is the number of units to use within each LSTM layer. Reducing this will improve speed and reduce memory usage, but can make compression rate worse. Make this a multiple of 8 to improve speed on certain GPUs._

#@markdown ---
num_layers = 8 #@param {type:"integer"}
#@markdown _This is the number of LSTM layers to use. Reducing this will improve speed, but can make compression rate worse._

#@markdown ---
embedding_size=512 #@param {type:"integer"}
#@markdown _Size of the embedding layer._

#@markdown ---
ensemble_size = 1 #@param {type:"integer"}
#@markdown _Number of LSTM models to use in the ensemble._

#@markdown ---
learning_rate_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _Learning rate schedule (step:value). linearly interpolate the value between points. special handling for first/last points: from start of file to the first point, keep it constant. Similarly, from the last data point to the end of the file it should remain constant at that value._

#@markdown ---
adam_b1 = 0.0 #@param {type:"number"}
#@markdown _Beta1 for Adam optimizer._

#@markdown ---
adam_b2 = 0.9999 #@param {type:"number"}
#@markdown _Beta2 for Adam optimizer._

#@markdown ---
adam_eps = 1e-12 #@param {type:"number"}
#@markdown _Epsilon for Adam optimizer._

#@markdown ---
mode = 'compress' #@param ["compress", "decompress", "both", "preprocess_only"]
#@markdown _Whether to run compression only, decompression only, or both. "preprocess_only" will only run preprocessing and skip compression._

#@markdown ---
preprocess = 'nncp' #@param ["cmix", "nncp", "nncp-done", "none"]
#@markdown _The choice of preprocessor. NNCP works better on enwik8/enwik9. NNCP preprocessing is slower since it constructs a custom dictionary, while cmix uses a pretrained dictionary. "nncp_done" is used for files which have already been preprocessed by NNCP (the dictionary must also be included)._

#@markdown ---
n_words = 8192 #@param {type:"integer"}
#@markdown _Only used for NNCP preprocessor: this is the approximative maximum number of words of the dictionary. Recommended value for enwik8/enwik9: 8192._

#@markdown ---
min_freq = 64 #@param {type:"integer"}
#@markdown _Only used for NNCP preprocessor: this is the minimum frequency of the selected words. Recommended value for enwik8: 64, enwik9: 512._

#@markdown ---
path_to_file = "enwik8" #@param ["enwik4", "enwik6", "enwik8", "enwik9", "custom"]
#@markdown _Name of the file to compress or decompress. If "custom" is selected, use the next parameter to set a custom path._

#@markdown ---
custom_path = '' #@param {type:"string"}
#@markdown _Use this if the previous parameter was set to "custom". Set this to the name of the file you want to compress/decompress. You can transfer files using the "http_path" or "local_upload" options below._

#@markdown ---
http_path = '' #@param {type:"string"}
#@markdown _The file from this URL will be downloaded. It is recommended to use Google Drive URLs to get fast transfer speed. Use this format for Google Drive files: https://drive.google.com/uc?id= and paste the file ID at the end of the URL. You can find the file ID from the "Get Link" URL in Google Drive. You can enter multiple URLs here, space separated._

#@markdown ---
local_upload = False #@param {type:"boolean"}
#@markdown _If enabled, you will be prompted in the "Setup Files" section to select files to upload from your local computer. You can upload multiple files. Note: the upload speed can be quite slow (use "http_path" for better transfer speeds)._

#@markdown ---
download_option = "no_download" #@param ["no_download", "local", "google_drive"]
#@markdown _If this is set to "local", the output files will be downloaded to your computer after compression/decompression. If set to "google_drive", they will be copied to your Google Drive account (which is significantly faster than downloading locally)._

#@markdown ---
checkpoint = False #@param {type:"boolean"}
#@markdown _If this is enabled, a checkpoint of the model weights will be downloaded (using the "download_option" parameter). This can be useful for getting around session time limits for Colab, by splitting files into multiple segments and saving/loading the model weights between each segment. Checkpoints (if present) will automatically be loaded when starting compression._

#@markdown ---
total_parts = 1 #@param {type:"integer"}
#@markdown _Total number of parts to split the file processing into (across multiple sessions)._

#@markdown ---
current_part = 1 #@param {type:"integer"}
#@markdown _Current part number to process (1 to total_parts)._

#@markdown ---
retrain_period_schedule = "0:1001 200000:5001" #@param {type:"string"}
#@markdown _Retrain period schedule (step:value). linearly interpolate the value between points._

#@markdown ---
retrain_block_len = 100000 #@param {type:"integer"}
#@markdown _Retrain over last M symbols._

#@markdown ---
retrain_seq_length = 100 #@param {type:"integer"}
#@markdown _Sequence length to use for retraining. This allows using a longer horizon during retraining steps than during online compression._

#@markdown ---
retrain_batch_size = 256 #@param {type:"integer"}
#@markdown _Batch size to use for retraining. Increasing this improves parallelism during retraining._

#@markdown ---
retrain_lr_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _Retraining learning rate schedule (step:value). linearly interpolate the value between points._

#@markdown ---
retrain_dropout = 0.4 #@param {type:"number"}
#@markdown _Dropout rate used during retraining._

## Setup

In [ ]:
#@title Imports

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import random
from google.colab import files
import time
import math
import sys
import subprocess
import contextlib
import os
from typing import Any
from google.colab import drive
import pickle
from flax.training import checkpoints as flax_checkpoints

# Set JAX to use 64-bit precision if needed, but for compression mixed/32 is likely intended.
# JAX defaults to 32-bit floats.
# os.environ['JAX_ENABLE_X64'] = '1'

In [ ]:
#@title System Info

def system_info():
  """Prints out system information."""
  gpu_info = !nvidia-smi
  gpu_info = '\n'.join(gpu_info)
  if gpu_info.find('failed') >= 0:
    print('Select the Runtime → "Change runtime type" menu to enable a GPU accelerator, ')
    print('and then re-execute this cell.')
  else:
    print(gpu_info)
  print("JAX version: ", jax.__version__)
  !lscpu |grep 'Model name'
  !cat /proc/meminfo | head -n 3

system_info()

In [ ]:
#@title Mount Google Drive
if download_option == "google_drive":
  drive.mount('/content/gdrive')

In [ ]:
#@title Setup Files

!mkdir -p "data"

if local_upload:
  %cd data
  files.upload()
  %cd ..

if path_to_file == 'enwik8' or path_to_file == 'enwik6' or path_to_file == 'enwik4':
  %cd data
  !gdown --id 11-twesB-vGexGZpVSFbaCZDXKh5jMmMd
  !head -c 1000000 enwik8 > enwik6
  !head -c 10000 enwik8 > enwik4
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'enwik9':
  %cd data
  !gdown --id 1D2gCmf9AlXIBP62ARhy0XcIuIolOTRAE
  !unzip enwik9.zip
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'custom':
  path_to_file = 'data/' + custom_path

if http_path:
  %cd data
  paths = http_path.split()
  for path in paths:
    !gdown $path
  %cd ..

if preprocess == 'cmix':
  !gdown --id 1qa7K28tlUDs9GGYbaL_iE9M4m0L1bYm9
  !unzip cmix-v18.zip
  %cd cmix
  !make
  %cd ..

if preprocess == 'nncp' or preprocess == 'nncp-done':
  !gdown --id 1EzVPbRkBIIbgOzvEMeM0YpibDi2R4SHD
  !tar -xf nncp-2019-11-16.tar.gz
  %cd nncp-2019-11-16/
  !make preprocess
  %cd ..

if checkpoint:
    ckpt_path = "data/ckpt.zip"
    if os.path.exists(ckpt_path):
        print(f"Unzipping checkpoint from {ckpt_path}...")
        !unzip -o {ckpt_path} -d data/ckpt
    else:
        print(f"Checkpoint enabled but {ckpt_path} not found. Please upload it if resuming.")

if current_part > 1:
    partial_compressed_path = "data/compressed.dat"
    if not os.path.exists(partial_compressed_path):
        print(f"Warning: Resuming compression (part {current_part}) but {partial_compressed_path} not found.")
        print("Please ensure you have uploaded the partial compressed file from the previous session.")
        # If the user uploaded it with a different name, standardise it.
        # But we don't know the name.
    else:
        print(f"Found partial compressed file: {partial_compressed_path}")

In [ ]:
#@title Model Architecture

class LSTMModel(nn.Module):
  vocab_size: int
  embedding_size: int
  rnn_units: int
  num_layers: int
  dropout_rate: float = 0.0
  dtype: Any = jnp.float32

  @nn.compact
  def __call__(self, inputs, states, deterministic=True, return_sequence=False):
    """
    Args:
      inputs: (batch, seq_length) int32
      states: list of tuples (c, h), length num_layers. Each c/h is (batch, units).
              Note: Flax LSTMCell uses (carry, hidden) where carry is (c, h).
      deterministic: bool, if False, dropout is applied.
      return_sequence: bool, if True, returns logits for the entire sequence.
    """
    # Embed
    x = nn.Embed(num_embeddings=self.vocab_size, features=self.embedding_size, dtype=self.dtype)(inputs)
    x = nn.Dropout(rate=self.dropout_rate, deterministic=deterministic)(x)
    # x: (batch, seq, embed)
    
    # Transpose for scan over time: (seq, batch, embed)
    embedding_t = jnp.transpose(x, (1, 0, 2))
    
    new_states = []
    layer_outputs = [] # To store (seq, batch, units) for each layer
    
    curr_input = embedding_t
    
    # Create the Scanned LSTM Cell Class
    # We scan over axis 0 of the input (time dimension)
    LSTMScan = nn.scan(nn.LSTMCell,
                       variable_broadcast="params",
                       split_rngs={"params": False},
                       in_axes=0, out_axes=0)

    for i in range(self.num_layers):
      # Instantiate the scanned LSTM layer
      lstm_scan = LSTMScan(self.rnn_units, dtype=self.dtype)
      
      carry = states[i] # (c, h)
      
      # Run scan
      new_carry, outputs = lstm_scan(carry, curr_input)
      
      # Dropout on outputs
      if i < self.num_layers - 1:
          outputs = nn.Dropout(rate=self.dropout_rate, deterministic=deterministic)(outputs)

      new_states.append(new_carry)
      layer_outputs.append(outputs)
      
      # Next layer input: Concat(Embed, PreviousOutput)
      # Both are (seq, batch, ...)
      if i < self.num_layers - 1:
          curr_input = jnp.concatenate([embedding_t, outputs], axis=-1)
          
    # Output logic
    if return_sequence:
        if self.num_layers == 1:
            final_rep = layer_outputs[0] # (seq, batch, units)
        else:
            final_rep = jnp.concatenate(layer_outputs, axis=-1) # (seq, batch, units * num_layers)
        
        # Dense -> Softmax
        logits = nn.Dense(self.vocab_size, name='dense_logits', dtype=self.dtype)(final_rep)
        # logits: (seq, batch, vocab) -> transpose to (batch, seq, vocab)
        logits = jnp.transpose(logits, (1, 0, 2))
    else:
        # Take last timestep of each layer. 
        # layer_outputs[i] is (seq, batch, units), so [-1] is (batch, units).
        last_timesteps = [out[-1] for out in layer_outputs] 
        
        if self.num_layers == 1:
          final_rep = last_timesteps[0]
        else:
          final_rep = jnp.concatenate(last_timesteps, axis=-1)
        
        # Dense -> Softmax
        logits = nn.Dense(self.vocab_size, name='dense_logits', dtype=self.dtype)(final_rep)
    
    # We return logits for better numerical stability in loss function.
    # Probabilities can be computed via jax.nn.softmax(logits).
    # Cast mainly for mixed precision stability
    logits = logits.astype(jnp.float32)
    
    return logits, new_states


In [ ]:
#@title Compression Library

from functools import partial
from flax.serialization import to_bytes, from_bytes

def parse_schedule(schedule_str):
  """Parses a schedule string "step:value step:value" into a list of (step, value) tuples."""
  points = []
  try:
    for item in schedule_str.split():
      step, value = item.split(':')
      points.append((float(step), float(value)))
    points.sort(key=lambda x: x[0])
  except ValueError:
    print(f"Error parsing schedule: {schedule_str}")
    return []
  return points

def get_scheduled_value(schedule, step):
  """Linearly interpolates value for a given step based on the schedule."""
  if not schedule:
    return 0.0
  
  if step <= schedule[0][0]:
    return schedule[0][1]
  
  if step >= schedule[-1][0]:
    return schedule[-1][1]
  
  # Linear search for the interval (optimized via bisect if needed, but linear is fine for small schedules)
  for i in range(len(schedule) - 1):
    start_step, start_val = schedule[i]
    end_step, end_val = schedule[i+1]
    
    if start_step <= step <= end_step:
      fraction = (step - start_step) / (end_step - start_step)
      return start_val + fraction * (end_val - start_val)
      
  return schedule[-1][1] # Should be unreachable

def get_symbol(index, length, freq, coder, compress, data):
  """Runs arithmetic coding and returns the next symbol.

  Args:
    index: Int, position of the symbol in the file.
    length: Int, size limit of the file.
    freq: ndarray, predicted symbol probabilities.
    coder: this is the arithmetic coder.
    compress: Boolean, True if compressing, False if decompressing.
    data: List containing each symbol in the file.

  Returns:
    The next symbol, or 0 if "index" is over the file size limit.
  """
  symbol = 0
  if index < length:
    if compress:
      symbol = data[index]
      coder.write(freq, symbol)
    else:
      symbol = coder.read(freq)
      data[index] = symbol
  return symbol

def reset_seed():
  """Initializes various random seeds to help with determinism."""
  SEED = 1234
  os.environ['PYTHONHASHSEED']=str(SEED)
  random.seed(SEED)
  np.random.seed(SEED)

def download(path):
  """Downloads the file at the specified path."""
  if download_option == 'local':
    files.download(path)
  elif download_option == 'google_drive':
    !cp -f $path /content/gdrive/My\ Drive

@partial(jax.jit, static_argnames=['model'])
def predict_step(model, params, inputs, states):
    """Runs forward pass to get probabilities with ensemble geometric mean."""
    def forward_single(p, s):
        # Deterministic=True for prediction (no dropout)
        logits, new_s = model.apply(p, inputs, s, deterministic=True, return_sequence=False)
        return jax.nn.log_softmax(logits), new_s
    
    # vmap over ensemble dimension (axis 0 of params and states)
    # Returns (ensemble, batch, vocab) and (ensemble, layers, (c,h), batch, units) technically
    # But flax scan structure needs care.
    # States structure: list of tuples. Vmap will vectorise the leaves.
    log_probs_ensemble, new_states = jax.vmap(forward_single, in_axes=(0, 0))(params, states)
    
    # Geometric mean probabilities (normalized) represents softmax of mean log_probs
    return jax.nn.softmax(jnp.mean(log_probs_ensemble, axis=0)), new_states

@partial(jax.jit, static_argnames=['model', 'optimizer'])
def update_step(model, optimizer, params, opt_state, inputs, states, symbols, mask):
    """Runs forward+backward pass to update ensemble models and get next states."""
    
    def single_update(p, opt_s, s):
        def loss_func(params_single):
            # Deterministic=True for online update (no dropout)
            l_logits, l_states = model.apply(params_single, inputs, s, deterministic=True, return_sequence=False)
            one_hot = jax.nn.one_hot(symbols, model.vocab_size)
            # optax returns negative log
            loss_vals = optax.softmax_cross_entropy(l_logits, one_hot)
            # Return log_probs (masked by loss calculation elsewhere, but here we need raw probs for ensemble)
            return jnp.sum(loss_vals * mask), (l_states, jax.nn.log_softmax(l_logits))
            
        (loss_val, (new_s, log_probs)), grads = jax.value_and_grad(loss_func, has_aux=True)(p)
        
        updates, new_opt_s = optimizer.update(grads, opt_s, p)
        new_p = optax.apply_updates(p, updates)
        return new_p, new_opt_s, new_s, log_probs

    # vmap over ensemble dimension
    new_params, new_opt_state, new_states, all_log_probs = jax.vmap(
        single_update, in_axes=(0, 0, 0)
    )(params, opt_state, states)
    
    # Calculate ensemble loss (Geometric Mean of Probabilities)
    # This corresponds to Softmax of Mean Log Probs
    mean_log_probs = jnp.mean(all_log_probs, axis=0) # (batch, vocab)
    
    one_hot = jax.nn.one_hot(symbols, model.vocab_size)
    # Cross entropy of the normalized geometric mean ensemble
    loss_vals = optax.softmax_cross_entropy(mean_log_probs, one_hot)
    
    denom = jnp.sum(mask)
    return new_params, new_opt_state, new_states, jnp.sum(loss_vals * mask), denom

@partial(jax.jit, static_argnames=['model', 'optimizer'])
def retrain_step(model, optimizer, params, opt_state, inputs, targets, states, rngs):
    """Runs forward+backward pass over a sequence with dropout enabled."""
    
    def single_retrain(p, opt_s, s, rng):
        def loss_func(params_single):
            # Deterministic=False (Enable Dropout)
            # return_sequence=True (Use labels from entire sequence)
            l_logits, l_states = model.apply(params_single, inputs, s, 
                                           deterministic=False, return_sequence=True, 
                                           rngs={'dropout': rng})
                                           
            one_hot = jax.nn.one_hot(targets, model.vocab_size)
            # l_logits: (batch, seq, vocab), targets: (batch, seq)
            loss_vals = optax.softmax_cross_entropy(l_logits, one_hot)
            return jnp.mean(loss_vals), l_states
            
        (loss_val, new_s), grads = jax.value_and_grad(loss_func, has_aux=True)(p)
        
        updates, new_opt_s = optimizer.update(grads, opt_s, p)
        new_p = optax.apply_updates(p, updates)
        return new_p, new_opt_s, new_s, loss_val

    # vmap over ensemble dimension
    new_params, new_opt_state, new_states, losses = jax.vmap(
        single_retrain, in_axes=(0, 0, 0, 0)
    )(params, opt_state, states, rngs)
    
    return new_params, new_opt_state, new_states, jnp.mean(losses)

def process(compress, length, vocab_size, coder, data):
  """This runs compression/decompression.

  Args:
    compress: Boolean, True if compressing, False if decompressing.
    length: Int, size limit of the file.
    vocab_size: Int, size of the vocabulary.
    coder: this is the arithmetic coder.
    data: List containing each symbol in the file.
  """
  start = time.time()
  last_print_time = start
  reset_seed()

  # Parse Schedules
  lr_schedule = parse_schedule(learning_rate_schedule)
  retrain_p_schedule = parse_schedule(retrain_period_schedule)
  retrain_l_schedule = parse_schedule(retrain_lr_schedule)
  
  # Parameter Summary
  print(f"batch_size={batch_size}, seq_length={seq_length}, rnn_units={rnn_units}, num_layers={num_layers}, "
        f"embedding_size={embedding_size}, ensemble_size={ensemble_size}, learning_rate_schedule={learning_rate_schedule}, "
        f"adam_b1={adam_b1}, adam_b2={adam_b2}, adam_eps={adam_eps}, "
        f"retrain_period_schedule={retrain_period_schedule}, "
        f"retrain_block_len={retrain_block_len}, retrain_seq_length={retrain_seq_length}, "
        f"retrain_batch_size={retrain_batch_size}, retrain_lr_schedule={retrain_lr_schedule}, "
        f"retrain_dropout={retrain_dropout}, total_parts={total_parts}, current_part={current_part}")
  
  # Initialize JAX RNG
  rng_key = jax.random.PRNGKey(1234)
  rng_key, init_key, retrain_key = jax.random.split(rng_key, 3)
  
  # Use bfloat16 for TPU efficiency
  model_dtype = jnp.bfloat16

  model = LSTMModel(vocab_size=vocab_size, embedding_size=embedding_size, 
                    rnn_units=rnn_units, num_layers=num_layers, 
                    dropout_rate=retrain_dropout, dtype=model_dtype)
  
  # Initialize Model Parameters
  dummy_input = jnp.zeros((batch_size, seq_length), dtype=jnp.int32)
  # Flax LSTMCell state is (carry, hidden).
  # If model is in bfloat16, carry/hidden should be castable or handled.
  # We initialize with float32 zeros, Flax/JAX handles dtype promotion/cast in layer.
  dummy_states = [(jnp.zeros((batch_size, rnn_units)), jnp.zeros((batch_size, rnn_units))) for _ in range(num_layers)]
  
  # Ensemble Initialization
  keys = jax.random.split(init_key, ensemble_size)
  
  def init_single(key):
     return model.init(key, dummy_input, dummy_states)
     
  params = jax.vmap(init_single)(keys)
  
  # Check for checkpoint to restore
  if checkpoint:
      ckpt_dir = os.path.abspath('data/ckpt')
      # Flax checkpoint restores the structure if target is provided.
      # If no checkpoint exists, it returns target (params).
      params = flax_checkpoints.restore_checkpoint(ckpt_dir=ckpt_dir, target=params)
      # Note: restore_checkpoint does not fail if no checkpoint is found (unless step is specified),
      # it simply returns the target structure.
      # We check if a checkpoint was actually restored by looking at folder content for logging
      if flax_checkpoints.latest_checkpoint(ckpt_dir):
          print("Restored model weights from checkpoint.")
      else:
          print("No checkpoint found. Starting from scratch.")

  # Model Summary
  print("\n" + "="*80)
  print(f"Model Architecture (Ensemble Size: {ensemble_size})")
  print("="*80)
  
  # Print summary of a single model using tabulate
  # Note: tabulate was added in Flax 0.4.1. If not available, we skip.
  try:
      # Use a dummy key for tabulation
      tabulate_fn = nn.tabulate(model, jax.random.PRNGKey(0), console_kwargs={'width': 100})
      print(tabulate_fn(dummy_input, [(jnp.zeros((batch_size, rnn_units)), jnp.zeros((batch_size, rnn_units))) for _ in range(num_layers)]))
  except Exception as e:
      print(f"Could not print detailed summary table: {e}")

  # Total Parameters for Ensemble
  total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
  print(f"\nTotal Ensemble Parameters: {total_params:,}")
  print("="*80 + "\n")
  
  # Replicate dummy_states for ensemble
  dummy_states_single = dummy_states
  dummy_states = jax.tree.map(lambda x: jnp.broadcast_to(x, (ensemble_size,) + x.shape), dummy_states)
  
  # Optimizer
  split = math.ceil(length / batch_size)

  if lr_schedule:
      lr_steps = jnp.array([p[0] for p in lr_schedule], dtype=jnp.float32)
      lr_values = jnp.array([p[1] for p in lr_schedule], dtype=jnp.float32)
      scheduler = lambda step: jnp.interp(step, lr_steps, lr_values, left=lr_values[0], right=lr_values[-1])
  else:
      scheduler = lambda step: 0.0005 # Fallback
      
  optimizer = optax.chain(
      optax.clip_by_global_norm(4.0),
      optax.adam(learning_rate=scheduler, b1=adam_b1, b2=adam_b2, eps=adam_eps)
  )
  # Initialize optimizer state for ensemble
  opt_state = jax.vmap(optimizer.init)(params)

  # Retraining Optimizer
  # We will define the retraining optimizer dynamically inside the loop to support changing LR.
  # But we need an initial state.
  retrain_optimizer_def = lambda lr: optax.chain(
      optax.clip_by_global_norm(4.0),
      optax.adam(learning_rate=lr, b1=adam_b1, b2=adam_b2, eps=adam_eps)
  )
  retrain_start_lr = get_scheduled_value(retrain_l_schedule, 0) if retrain_l_schedule else 0.0005
  retrain_optimizer = retrain_optimizer_def(retrain_start_lr)
  retrain_opt_state = jax.vmap(retrain_optimizer.init)(params)
  
  # Define part steps
  part_len = math.ceil(split / total_parts)
  start_pos = (current_part - 1) * part_len
  end_pos = min(start_pos + part_len, split)
  print(f"Processing part {current_part} of {total_parts}. Step range: {start_pos} to {end_pos - 1} (Total split: {split})")
  
  # Initial symbols
  freq = np.cumsum(np.full(vocab_size, (1.0 / vocab_size)) * 10000000 + 1)
  
  pos = 0
  
  # State Loading / Warmup
  current_states = dummy_states
  
  model_state_loaded = False
  if start_pos > 0 and checkpoint:
      ckpt_dir = os.path.abspath('data/ckpt')
      model_state_path = os.path.join(ckpt_dir, 'model_state.pkl')
      
      if os.path.exists(model_state_path):
          print("Attempting to load model states from checkpoint...")
          try:
              with open(model_state_path, 'rb') as f:
                  loaded_state = pickle.load(f)
              
              # Load states_queue and seq_input
              # Note: We need to convert numpy arrays back to JAX arrays if they were saved as numpy
              # Or if they were saved as JAX, they might be on a different device?
              # pickle handles simple ndarrays fine, and JAX handles them too.
              
              states_queue_loaded = loaded_state['states_queue']
              seq_input_loaded = loaded_state['seq_input']
              
              # Verify shapes if possible. 
              # seq_input: (batch, seq_len)
              if seq_input_loaded.shape != (batch_size, seq_length):
                  print(f"Warning: Loaded seq_input shape {seq_input_loaded.shape} mismatch with batch/seq params.")
                  raise ValueError("Shape mismatch")
                  
              seq_input = jnp.array(seq_input_loaded)
              
              # states_queue: list of loop states.
              # Each item is tuple/list of states.
              # We just trust the pickle for now. Recursively convert to jnp?
              # The update loop needs valid JAX array structure.
              states_queue = jax.tree.map(jnp.array, states_queue_loaded)
              
              # Restore optimizers if available
              if 'opt_state_bytes' in loaded_state:
                  opt_state = from_bytes(opt_state, loaded_state['opt_state_bytes'])
                  print("Restored main optimizer state.")
              
              if 'retrain_opt_state_bytes' in loaded_state:
                  retrain_opt_state = from_bytes(retrain_opt_state, loaded_state['retrain_opt_state_bytes'])
                  print("Restored retraining optimizer state.")

              print(f"Successfully loaded model states. Skipping warmup.")
              model_state_loaded = True
              pos = start_pos
              
          except Exception as e:
              print(f"Failed to load model states: {e}. Falling back to warmup.")
  
  if start_pos > 0 and not model_state_loaded:
      warmup_len = 500
      run_start_pos = max(0, start_pos - warmup_len)
      
      # Re-build `seq_input` for `run_start_pos`.
      w_symbols = []
      for i in range(batch_size):
           # current pos `run_start_pos`.
           start_idx_window = run_start_pos - seq_length + 1
           batch_syms = []
           for k in range(seq_length):
               idx_k = start_idx_window + k # Goes from `start` to `start + L - 1 = run_start_pos`
               s_idx = idx_k + i*split
               # Read data
               if s_idx < 0:
                   # For i=0 stream, early index
                   val = data[i*split] if i*split < len(data) else 0 # Fallback
               else:
                   val = data[s_idx] if s_idx < len(data) else 0
               batch_syms.append(val)
           w_symbols.append(batch_syms)
      seq_input = jnp.array(w_symbols, dtype=jnp.int32)
      
      states_queue = [dummy_states] * seq_length # Reset
      
      # Run the warmup
      print(f"Warming up states from {run_start_pos} to {start_pos}...")
      for w_pos in range(run_start_pos, start_pos):
           state_in = states_queue.pop(0)
           _, new_states = predict_step(model, params, seq_input, state_in)
           states_queue.append(new_states)
           
           # Update input window
           next_in_symbols = []
           for i in range(batch_size):
               # Next symbol index: w_pos + 1 + i*split
               idx = w_pos + 1 + i * split
               val = data[idx] if idx < len(data) else 0
               next_in_symbols.append(val)
           next_in_jax = jnp.expand_dims(jnp.array(next_in_symbols, dtype=jnp.int32), 1)
           seq_input = jnp.concatenate([seq_input[:, 1:], next_in_jax], axis=1)

      pos = start_pos
      
  elif not model_state_loaded:
      # Default initialization
      initial_symbols = []
      for i in range(batch_size):
        initial_symbols.append(get_symbol(i*split, length, freq, coder, compress, data))
      seq_input = jnp.tile(jnp.expand_dims(jnp.array(initial_symbols, dtype=jnp.int32), 1), (1, seq_length))
      states_queue = [dummy_states] * seq_length
      pos = 0

  cross_entropy = 0.0 
  denom = 0.0
  last_output = 0
  template = '{:0.2f}%\tcross entropy: {:0.2f}\ttime: {:0.2f}\tlr: {:0.8f}\tstep: {}'
  
  # Pre-compile functions with static args
  predict_fn = partial(predict_step, model)
  update_fn = partial(update_step, model, optimizer)
  
  # For retraining, we define the function dynamically inside the loop based on current LR.

  last_retrain_pos = 0

  while pos < end_pos:

    # Calculate current retrain params
    current_retrain_period = get_scheduled_value(retrain_p_schedule, pos)
    current_retrain_lr = get_scheduled_value(retrain_l_schedule, pos)

    # Check for retraining
    if current_retrain_period > 0 and (pos - last_retrain_pos) >= current_retrain_period:
        retrain_start_time = time.time()
        
        # Loop start for retraining
        r_start_step = max(0, pos - retrain_block_len)
        r_end_step = pos
        
        # We need to collect ALL training examples first
        # Format: (total_examples, seq_len)
        all_inputs = []
        all_targets = []
        
        r_step = r_start_step
        while r_step < r_end_step:
            for i in range(batch_size):
                base_idx = r_step + i * split 
                # Note: at `pos=0`, we loaded `data[0 + i*split]`.
                # Loop starts at `pos=0`. inside loop `index = pos + 1 + i * split`.
                # So data is filled at `0`, `1`, `2`... relative to `i*split`.
                # So for stream `i`, data is valid from `i*split` to `i*split + pos + 1`.
                
                start_idx = base_idx
                end_idx = start_idx + retrain_seq_length + 1
                current_stream_limit = i * split + pos + 1

                stream_segment = data[start_idx : min(end_idx, current_stream_limit)]
                
                # We need exact retrain_seq_length + 1 symbols.
                # If segment is too short, we pad. This happens at the very end of stream.
                if len(stream_segment) < retrain_seq_length + 1:
                    stream_segment += [0] * (retrain_seq_length + 1 - len(stream_segment))
                
                all_inputs.append(stream_segment[:-1])
                all_targets.append(stream_segment[1:])
            
            r_step += retrain_seq_length
            
        # Convert to JAX arrays
        # Shape: (total_segments, seq_len)
        all_inputs_jax = jnp.array(all_inputs, dtype=jnp.int32)
        all_targets_jax = jnp.array(all_targets, dtype=jnp.int32)
        
        total_examples = all_inputs_jax.shape[0]
        
        # Skip training on the remainder if not divisible by retrain_batch_size to avoid JIT recompilation
        remainder = total_examples % retrain_batch_size
        if remainder != 0:
            # Drop the last 'remainder' examples
            all_inputs_jax = all_inputs_jax[:-remainder]
            all_targets_jax = all_targets_jax[:-remainder]
            total_examples -= remainder

        print(f"Starting retraining at step {pos}... (period={current_retrain_period:.1f}, lr={current_retrain_lr:.8f}, examples={total_examples})")

        # Update Retraining Optimizer with current LR
        curr_retrain_opt = retrain_optimizer_def(current_retrain_lr)
        curr_retrain_fn = partial(retrain_step, model, curr_retrain_opt)


        if total_examples > 0:
            # Iterate over super-batches
            for i in range(0, total_examples, retrain_batch_size):
                batch_inputs = all_inputs_jax[i : i + retrain_batch_size]
                batch_targets = all_targets_jax[i : i + retrain_batch_size]
                
                # Initial states for this batch (zeros)
                # Shape for one layer state component: (ensemble_size, retrain_batch_size, rnn_units)
                batch_states = [
                    (jnp.zeros((ensemble_size, retrain_batch_size, rnn_units)), 
                     jnp.zeros((ensemble_size, retrain_batch_size, rnn_units))) 
                    for _ in range(num_layers)
                ]
                                
                # Generate dropout keys
                retrain_key, dropout_key = jax.random.split(retrain_key)
                dropout_keys = jax.random.split(dropout_key, ensemble_size)

                params, retrain_opt_state, _, r_loss = curr_retrain_fn(
                    params, retrain_opt_state, batch_inputs, batch_targets, batch_states, dropout_keys
                )

        last_retrain_pos = pos
        print(f"Retraining finished. Duration: {time.time() - retrain_start_time:.2f}s")

    state_in = states_queue.pop(0)

    # 1. Predict
    probs, _ = predict_fn(params, seq_input, state_in)
    
    # 2. Arithmetic Coding (CPU)
    # Convert optimized JAX array to numpy for CPU processing
    p = np.array(probs)
    
    current_symbols = []
    current_mask = []
    
    for i in range(batch_size):
        p_i = p[i]
        freq = np.cumsum(p_i * 10000000 + 1)
        index = pos + 1 + i * split
        
        symbol = get_symbol(index, length, freq, coder, compress, data)
        current_symbols.append(symbol)
        
        if index < length:
            current_mask.append(1.0)
        else:
            current_mask.append(0.0)
            
    symbols_jax = jnp.array(current_symbols, dtype=jnp.int32)
    
    # 3. Update Model
    mask_jax = jnp.array(current_mask, dtype=jnp.float32)

    params, opt_state, new_states, loss_val, loss_denom = update_fn(
        params, opt_state, seq_input, state_in, symbols_jax, mask_jax
    )
    
    # 4. Update State Queue
    states_queue.append(new_states)
    
    # 5. Update Metrics
    cross_entropy += loss_val
    denom += loss_denom
    
    # 6. Prepare for next step
    # Shift seq_input: discard first col, append new symbol
    # seq_input shape: (batch, seq_length)
    # symbols_jax shape: (batch,) -> expand to (batch, 1)
    
    seq_input = jnp.concatenate([seq_input[:, 1:], jnp.expand_dims(symbols_jax, 1)], axis=1)

    pos += 1
    
    if time.time() - last_print_time >= 20:
       last_print_time = time.time()
       time_diff = last_print_time - start
       current_bpc = (cross_entropy / denom) / np.log(2)
       current_lr = scheduler(pos)
       print(template.format(pos / split * 100, current_bpc, time_diff, current_lr, pos))
  
  if checkpoint:
      # Save checkpoint
      print("Saving checkpoint...")
      
      ckpt_dir = os.path.abspath('data/ckpt')
      if not os.path.exists(ckpt_dir):
        os.makedirs(ckpt_dir)
      
      flax_checkpoints.save_checkpoint(ckpt_dir=ckpt_dir, target=params, step=0, overwrite=True)
      print("Checkpoint saved.")
      
      # Save AC state as well!
      # We just pickle the dict returned by get_state()
      ac_state = {
          'coder': coder.get_state(),
          'bitstream': coder.output.get_state() if compress else coder.input.get_state()
      }
      
      # If decompressing, we need input file position!
      if not compress:
           # coder.input is BitInputStream
           # coder.input.input is the file object
           ac_state['file_pos'] = coder.input.input.tell()
           
      with open(os.path.join(ckpt_dir, 'ac_state.pkl'), 'wb') as f:
          pickle.dump(ac_state, f)
      print("AC state saved.")
      
      # Save Model States for better resumption
      # REMOVED Check "if compress:" to allow saving state during decompression as well.
      # This is crucial for splitting decompression sessions without massive warmup.
      
      # Convert JAX structures to numpy for pickling
      # seq_input is simple array
      # states_queue is list of lists of tuple...
      # recursively map to np
      seq_input_np = np.array(seq_input)
      states_queue_np = jax.tree.map(np.array, states_queue)
      
      # Serialize Optimizer States using Flax serialization
      # pickle generally works, but Flax serialization is more robust for JAX types.
      # opt_state is a PyTree (Optax state).
      
      opt_state_bytes = to_bytes(opt_state)
      retrain_opt_state_bytes = to_bytes(retrain_opt_state)
      
      model_state = {
          'seq_input': seq_input_np,
          'states_queue': states_queue_np,
          'opt_state_bytes': opt_state_bytes,
          'retrain_opt_state_bytes': retrain_opt_state_bytes
      }
      with open(os.path.join(ckpt_dir, 'model_state.pkl'), 'wb') as f:
          pickle.dump(model_state, f)
      print("Model state (including optimizers) saved.")

In [ ]:
#@title Arithmetic Coding Library

#
# Reference arithmetic coding
# Copyright (c) Project Nayuki
#
# https://www.nayuki.io/page/reference-arithmetic-coding
# https://github.com/nayuki/Reference-arithmetic-coding
#

import sys
python3 = sys.version_info.major >= 3


# ---- Arithmetic coding core classes ----

# Provides the state and behaviors that arithmetic coding encoders and decoders share.
class ArithmeticCoderBase(object):

	# Constructs an arithmetic coder, which initializes the code range.
	def __init__(self, numbits):
		if numbits < 1:
			raise ValueError("State size out of range")

		# -- Configuration fields --
		# Number of bits for the 'low' and 'high' state variables. Must be at least 1.
		# - Larger values are generally better - they allow a larger maximum frequency total (maximum_total),
		#   and they reduce the approximation error inherent in adapting fractions to integers;
		#   both effects reduce the data encoding loss and asymptotically approach the efficiency
		#   of arithmetic coding using exact fractions.
		# - But larger state sizes increase the computation time for integer arithmetic,
		#   and compression gains beyond ~30 bits essentially zero in real-world applications.
		# - Python has native bigint arithmetic, so there is no upper limit to the state size.
		#   For Java and C++ where using native machine-sized integers makes the most sense,
		#   they have a recommended value of num_state_bits=32 as the most versatile setting.
		self.num_state_bits = numbits
		# Maximum range (high+1-low) during coding (trivial), which is 2^num_state_bits = 1000...000.
		self.full_range = 1 << self.num_state_bits
		# The top bit at width num_state_bits, which is 0100...000.
		self.half_range = self.full_range >> 1  # Non-zero
		# The second highest bit at width num_state_bits, which is 0010...000. This is zero when num_state_bits=1.
		self.quarter_range = self.half_range >> 1  # Can be zero
		# Minimum range (high+1-low) during coding (non-trivial), which is 0010...010.
		self.minimum_range = self.quarter_range + 2  # At least 2
		# Maximum allowed total from a frequency table at all times during coding. This differs from Java
		# and C++ because Python's native bigint avoids constraining the size of intermediate computations.
		self.maximum_total = self.minimum_range
		# Bit mask of num_state_bits ones, which is 0111...111.
		self.state_mask = self.full_range - 1

		# -- State fields --
		# Low end of this arithmetic coder's current range. Conceptually has an infinite number of trailing 0s.
		self.low = 0
		# High end of this arithmetic coder's current range. Conceptually has an infinite number of trailing 1s.
		self.high = self.state_mask
        
	def get_state(self):
		return {
			'low': self.low,
			'high': self.high
		}
	
	def set_state(self, state):
		self.low = state['low']
		self.high = state['high']


	# Updates the code range (low and high) of this arithmetic coder as a result
	# of processing the given symbol with the given frequency table.
	# Invariants that are true before and after encoding/decoding each symbol
	# (letting full_range = 2^num_state_bits):
	# - 0 <= low <= code <= high < full_range. ('code' exists only in the decoder.)
	#   Therefore these variables are unsigned integers of num_state_bits bits.
	# - low < 1/2 * full_range <= high.
	#   In other words, they are in different halves of the full range.
	# - (low < 1/4 * full_range) || (high >= 3/4 * full_range).
	#   In other words, they are not both in the middle two quarters.
	# - Let range = high - low + 1, then full_range/4 < minimum_range
	#   <= range <= full_range. These invariants for 'range' essentially
	#   dictate the maximum total that the incoming frequency table can have.
	def update(self, freqs, symbol):
		# State check
		low = self.low
		high = self.high
		# if low >= high or (low & self.state_mask) != low or (high & self.state_mask) != high:
		# 	raise AssertionError("Low or high out of range")
		range = high - low + 1
		# if not (self.minimum_range <= range <= self.full_range):
		# 	raise AssertionError("Range out of range")

		# Frequency table values check
		total = int(freqs[-1])
		symlow = int(freqs[symbol-1]) if symbol > 0 else 0
		symhigh = int(freqs[symbol])
		#total = freqs.get_total()
		#symlow = freqs.get_low(symbol)
		#symhigh = freqs.get_high(symbol)
		# if symlow == symhigh:
		# 	raise ValueError("Symbol has zero frequency")
		# if total > self.maximum_total:
		# 	raise ValueError("Cannot code symbol because total is too large")

		# Update range
		newlow  = low + symlow  * range // total
		newhigh = low + symhigh * range // total - 1
		self.low = newlow
		self.high = newhigh

		# While low and high have the same top bit value, shift them out
		while ((self.low ^ self.high) & self.half_range) == 0:
			self.shift()
			self.low  = ((self.low  << 1) & self.state_mask)
			self.high = ((self.high << 1) & self.state_mask) | 1
		# Now low's top bit must be 0 and high's top bit must be 1

		# While low's top two bits are 01 and high's are 10, delete the second highest bit of both
		while (self.low & ~self.high & self.quarter_range) != 0:
			self.underflow()
			self.low = (self.low << 1) ^ self.half_range
			self.high = ((self.high ^ self.half_range) << 1) | self.half_range | 1


	# Called to handle the situation when the top bit of 'low' and 'high' are equal.
	def shift(self):
		raise NotImplementedError()


	# Called to handle the situation when low=01(...) and high=10(...).
	def underflow(self):
		raise NotImplementedError()


# Encodes symbols and writes to an arithmetic-coded bit stream.
class ArithmeticEncoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding encoder based on the given bit output stream.
	def __init__(self, numbits, bitout):
		super(ArithmeticEncoder, self).__init__(numbits)
		# The underlying bit output stream.
		self.output = bitout
		# Number of saved underflow bits. This value can grow without bound.
		self.num_underflow = 0
		
	def get_state(self):
		state = super(ArithmeticEncoder, self).get_state()
		state['num_underflow'] = self.num_underflow
		return state
	
	def set_state(self, state):
		super(ArithmeticEncoder, self).set_state(state)
		self.num_underflow = state['num_underflow']


	# Encodes the given symbol based on the given frequency table.
	# This updates this arithmetic coder's state and may write out some bits.
	def write(self, freqs, symbol):
		self.update(freqs, symbol)


	# Terminates the arithmetic coding by flushing any buffered bits, so that the output can be decoded properly.
	# It is important that this method must be called at the end of the each encoding process.
	# Note that this method merely writes data to the underlying output stream but does not close it.
	def finish(self):
		self.output.write(1)


	def shift(self):
		bit = self.low >> (self.num_state_bits - 1)
		self.output.write(bit)

		# Write out the saved underflow bits
		for _ in range(self.num_underflow):
			self.output.write(bit ^ 1)
		self.num_underflow = 0


	def underflow(self):
		self.num_underflow += 1


# Reads from an arithmetic-coded bit stream and decodes symbols.
class ArithmeticDecoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding decoder based on the
	# given bit input stream, and fills the code bits.
	def __init__(self, numbits, bitin):
		super(ArithmeticDecoder, self).__init__(numbits)
		# The underlying bit input stream.
		self.input = bitin
		# The current raw code bits being buffered, which is always in the range [low, high].
		self.code = 0
		for _ in range(self.num_state_bits):
			self.code = self.code << 1 | self.read_code_bit()
	
	def get_state(self):
		state = super(ArithmeticDecoder, self).get_state()
		state['code'] = self.code
		return state
	
	def set_state(self, state):
		super(ArithmeticDecoder, self).set_state(state)
		self.code = state['code']


	# Decodes the next symbol based on the given frequency table and returns it.
	# Also updates this arithmetic coder's state and may read in some bits.
	def read(self, freqs):
		#if not isinstance(freqs, CheckedFrequencyTable):
		#	freqs = CheckedFrequencyTable(freqs)

		# Translate from coding range scale to frequency table scale
		total = int(freqs[-1])
		#total = freqs.get_total()
		#if total > self.maximum_total:
		#	raise ValueError("Cannot decode symbol because total is too large")
		range = self.high - self.low + 1
		offset = self.code - self.low
		value = ((offset + 1) * total - 1) // range
		#assert value * range // total <= offset
		#assert 0 <= value < total

		# A kind of binary search. Find highest symbol such that freqs.get_low(symbol) <= value.
		start = 0
		end = len(freqs)
		#end = freqs.get_symbol_limit()
		while end - start > 1:
			middle = (start + end) >> 1
			low = int(freqs[middle-1]) if middle > 0 else 0
			#if freqs.get_low(middle) > value:
			if low > value:
				end = middle
			else:
				start = middle
		#assert start + 1 == end

		symbol = start
		#assert freqs.get_low(symbol) * range // total <= offset < freqs.get_high(symbol) * range // total
		self.update(freqs, symbol)
		#if not (self.low <= self.code <= self.high):
		#	raise AssertionError("Code out of range")
		return symbol


	def shift(self):
		self.code = ((self.code << 1) & self.state_mask) | self.read_code_bit()


	def underflow(self):
		self.code = (self.code & self.half_range) | ((self.code << 1) & (self.state_mask >> 1)) | self.read_code_bit()


	# Returns the next bit (0 or 1) from the input stream. The end
	# of stream is treated as an infinite number of trailing zeros.
	def read_code_bit(self):
		temp = self.input.read()
		if temp == -1:
			temp = 0
		return temp


# ---- Bit-oriented I/O streams ----

# A stream of bits that can be read. Because they come from an underlying byte stream,
# the total number of bits is always a multiple of 8. The bits are read in big endian.
class BitInputStream(object):

	# Constructs a bit input stream based on the given byte input stream.
	def __init__(self, inp):
		# The underlying byte stream to read from
		self.input = inp
		# Either in the range [0x00, 0xFF] if bits are available, or -1 if end of stream is reached
		self.currentbyte = 0
		# Number of remaining bits in the current byte, always between 0 and 7 (inclusive)
		self.numbitsremaining = 0
  
	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsremaining': self.numbitsremaining
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsremaining = state['numbitsremaining']


	# Reads a bit from this stream. Returns 0 or 1 if a bit is available, or -1 if
	# the end of stream is reached. The end of stream always occurs on a byte boundary.
	def read(self):
		if self.currentbyte == -1:
			return -1
		if self.numbitsremaining == 0:
			temp = self.input.read(1)
			if len(temp) == 0:
				self.currentbyte = -1
				return -1
			self.currentbyte = temp[0] if python3 else ord(temp)
			self.numbitsremaining = 8
		assert self.numbitsremaining > 0
		self.numbitsremaining -= 1
		return (self.currentbyte >> self.numbitsremaining) & 1


	# Reads a bit from this stream. Returns 0 or 1 if a bit is available, or raises an EOFError
	# if the end of stream is reached. The end of stream always occurs on a byte boundary.
	def read_no_eof(self):
		result = self.read()
		if result != -1:
			return result
		else:
			raise EOFError()


	# Closes this stream and the underlying input stream.
	def close(self):
		self.input.close()
		self.currentbyte = -1
		self.numbitsremaining = 0


# A stream where bits can be written to. Because they are written to an underlying
# byte stream, the end of the stream is padded with 0's up to a multiple of 8 bits.
# The bits are written in big endian.
class BitOutputStream(object):

	# Constructs a bit output stream based on the given byte output stream.
	def __init__(self, out):
		self.output = out  # The underlying byte stream to write to
		self.currentbyte = 0  # The accumulated bits for the current byte, always in the range [0x00, 0xFF]
		self.numbitsfilled = 0  # Number of accumulated bits in the current byte, always between 0 and 7 (inclusive)

	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsfilled': self.numbitsfilled
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsfilled = state['numbitsfilled']

	# Writes a bit to the stream. The given bit must be 0 or 1.
	def write(self, b):
		if b not in (0, 1):
			raise ValueError("Argument must be 0 or 1")
		self.currentbyte = (self.currentbyte << 1) | b
		self.numbitsfilled += 1
		if self.numbitsfilled == 8:
			towrite = bytes((self.currentbyte,)) if python3 else chr(self.currentbyte)
			self.output.write(towrite)
			self.currentbyte = 0
			self.numbitsfilled = 0


	# Closes this stream and the underlying output stream. If called when this
	# bit stream is not at a byte boundary, then the minimum number of "0" bits
	# (between 0 and 7 of them) are written as padding to reach the next byte boundary.
	def close(self):
		while self.numbitsfilled != 0:
			self.write(0)
		self.output.close()

## Compress

In [ ]:
#@title Preprocess

if mode != 'decompress':
  input_path = path_to_file

  if preprocess == 'cmix':
    !./cmix/cmix -s ./cmix/dictionary/english.dic $path_to_file ./data/preprocessed.dat
    input_path = "./data/preprocessed.dat"

  # int_list will contain the characters of the file.
  int_list = []
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if preprocess == 'nncp':
      !time ./nncp-2019-11-16/preprocess c data/dictionary.words $path_to_file data/preprocessed.dat $n_words $min_freq
    else:
      !cp $path_to_file data/preprocessed.dat
    input_path = "./data/preprocessed.dat"
    orig = open(input_path, 'rb').read()
    for i in range(0, len(orig), 2):
      int_list.append(orig[i] * 256 + orig[i+1])
    vocab_size = int(subprocess.check_output(
        ['wc', '-l', 'data/dictionary.words']).split()[0])
  else:
    text = open(input_path, 'rb').read()
    vocab = sorted(set(text))
    vocab_size = len(vocab)
    # Creating a mapping from unique characters to indexes.
    char2idx = {u:i for i, u in enumerate(vocab)}
    for idx, c in enumerate(text):
      int_list.append(char2idx[c])

  # Round up to a multiple of 8 to improve performance.
  vocab_size = math.ceil(vocab_size/8) * 8
  
  if checkpoint:
      ckpt_dir = 'data/ckpt'
      if not os.path.exists(ckpt_dir):
          os.makedirs(ckpt_dir)
  else:
      ckpt_dir = None

  file_len = len(int_list)
  print ('Length of file: {} symbols'.format(file_len))
  print ('Vocabulary size: {}'.format(vocab_size))

In [ ]:
#@title Compression

if mode == 'compress' or mode == 'both':
  original_file = path_to_file
  path_to_file = "data/compressed.dat"
  
  # Determine open mode
  if current_part > 1:
      open_mode = "ab"
      print(f"Resuming compression (part {current_part}/{total_parts}). Appending to {path_to_file}...")
  else:
      open_mode = "wb"
      print(f"Starting compression (part {current_part}/{total_parts})...")

  with open(path_to_file, open_mode) as out:
      # If resuming, we don't create a new BitOutputStream right away if we need to restore state.
      # But BitOutputStream initialization is simple.
      bitout = BitOutputStream(out)
      
      enc = ArithmeticEncoder(32, bitout)

      # Restore AC state if resuming
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              enc.set_state(ac_state['coder'])
              bitout.set_state(ac_state['bitstream'])
              print("Restored AC state.")
          else:
              print("Warning: AC state not found for resume! This will likely fail.")

      length = len(int_list)
      
      # Write header only if starting fresh
      if current_part == 1:
          # Write the original file length to the compressed file header.
          out.write(length.to_bytes(5, byteorder='big', signed=False))
          if preprocess != 'nncp' and preprocess != 'nncp-done':
              # If NNCP was not used for preprocessing, write 256 bits to the compressed
              # file header to keep track of the vocabulary.
              for i in range(256):
                  if i in char2idx:
                      bitout.write(1)
                  else:
                      bitout.write(0)

      process(True, length, vocab_size, enc, int_list)
      
      # Finish only if this is the last part
      if current_part == total_parts:
          enc.finish()
          bitout.close() # Write padding
      else:
          # If pausing, we DO NOT close bitout to avoid writing padding bits.
          # The file `out` will be closed by the `with` block, flushing bytes.
          # Any partial bits in `bitout` are saved in pickle and not written to disk.
          print(f"Pausing compression part {current_part}. AC state saved (in process).")
          pass

  print("Compressed size:", os.path.getsize(path_to_file))

In [ ]:
#@title Download Result

if mode == 'preprocess_only':
  if preprocess == 'nncp':
    download('data/dictionary.words')
  download(input_path)
elif mode != 'decompress':
  download('data/compressed.dat')
  if preprocess == 'nncp':
    download('data/dictionary.words')
  if checkpoint and mode != "both":
    # flax.training.checkpoints creates files like "checkpoint_0"
    # We should zip or download the directory
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

## Decompress

In [ ]:
#@title Decompression

if mode == 'decompress' or mode == 'both':
  output_path = "data/decompressed.dat"
  
  # Check if we are resuming decompression
  if current_part > 1:
      if not os.path.exists(output_path):
          print(f"Error: Resuming decompression (part {current_part}), but {output_path} not found.")
          print("Please upload the partial decompressed file from the previous session.")
      else:
           print(f"Resuming decompression (part {current_part}/{total_parts}). Appending to {output_path}...")
  else:
      print(f"Starting decompression (part {current_part}/{total_parts})...")

  # We need to read the compressed file.
  # If resuming, we need to seek to the correct bit position?
  # The ArithmeticDecoder maintains state (code, low, high, input stream position).
  # We saved this state in `ac_state.pkl`.
  
  # However, `BitInputStream` caches bits. We need to restore `currentbyte` and `numbitsremaining`.
  # AND the underlying file pointer `inp` position.
  
  # The `ac_state` for `input` (BitInputStream) should arguably include the file position if we closed the file.
  # But `BitInputStream` wraps a file object. It doesn't know about file paths.
  # We can't pickle the file object.
  
  # Start reading
  with open(path_to_file, "rb") as inp:
      # Read the original file size from the header.
      length = int.from_bytes(inp.read()[:5], byteorder='big')
      
      # Prepare output buffer
      # If resuming, we need to load existing data into `output` list because `process`
      # needs it for context (warmup).
      # This might be memory intensive for large files, but `output` is list of ints.
      # enwik8 is 100MB, so 100MB ints is fine. enwik9 is 1GB, 1GB ints is 4GB or 8GB RAM.
      # This is acceptable on Colab High RAM.
      
      output = [0] * length
      if current_part > 1:
           # Load previous data from output_path
           print("Loading previously decompressed data for context...")
           if os.path.exists(output_path):
                # If NNCP preprocessing (split bytes), we need to reconstruct?
                # The `output` list in `process` holds 0-255 if no NNCP logic in `process`?
                # Wait, `process` works on `data` which is `int_list`.
                # In compression: `int_list` comes from `check_output` or reading file.
                # In decompression:
                # `output = [0] * length`
                # `process(..., output)`
                # Then `output` is written to file.
                
                # If we resume, we need `output` to contain the data up to `start_pos`.
                # We can read `output_path`.
                with open(output_path, "rb") as f_partial:
                    partial_data = f_partial.read()
                    
                # We need to map `partial_data` bytes back to `output` symbols.
                if preprocess == 'nncp' or preprocess == 'nncp-done':
                    # output[i] was written as high/low byte?
                    # "out.write(bytes(((output[i] // 256),)))"
                    # "out.write(bytes(((output[i] % 256),)))"
                    # So 2 bytes per symbol.
                    for i in range(len(partial_data) // 2):
                        val = partial_data[2*i] * 256 + partial_data[2*i+1]
                        output[i] = val
                else:
                    # Generic case: map char to index?
                    # "out.write(bytes((idx2char[output[i]],)))"
                    # We need the inverse mapping `char2idx` which was built in Compress cell?
                    # But Decompress cell reconstructs `vocab` from header.
                    pass # We handle vocab reconstruction below first.

      inp.seek(5)
      bitin = BitInputStream(inp)

      if preprocess == 'nncp' or preprocess == 'nncp-done':
        # If the preprocessor is NNCP, we can get the vocab_size from the
        # dictionary.
        vocab_size = int(subprocess.check_output(
            ['wc', '-l', 'data/dictionary.words']).split()[0])
      else:
        # If the preprocessor is not NNCP, we can get the vocabulary from the file
        # header.
        vocab = []
        for i in range(256):
          if bitin.read():
            vocab.append(i)
        vocab_size = len(vocab)
        
        # Determine char2idx if we need to reload partial data
        if current_part > 1 and not (preprocess == 'nncp' or preprocess == 'nncp-done'):
             # Reconstruct
             # vocab is list of chars (ints 0-255)
             # idx2char = np.array(vocab)
             # output[i] corresponds to index in vocab.
             # existing file contains chars.
             # so we need reverse map.
             char2val = {c: i for i, c in enumerate(vocab)}
             
             # Fill output
             for i in range(len(partial_data)):
                 c = partial_data[i]
                 if c in char2val:
                     output[i] = char2val[c]
                 else:
                     # This shouldn't happen if file corresponds to vocab
                     pass


      # Round up to a multiple of 8 to improve performance.
      vocab_size = math.ceil(vocab_size/8) * 8
      
      dec = ArithmeticDecoder(32, bitin)
      
      # Restore AC state if resuming
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              
              dec.set_state(ac_state['coder'])
              bitin.set_state(ac_state['bitstream'])
              
              # IMPORTANT: Restore file position!
              # The BitInputStream state does not include file position.
              # But we can infer it? No.
              # We should have saved it.
              
              if 'file_pos' in ac_state:
                  inp.seek(ac_state['file_pos'])
              else:
                   print("Warning: File position not found in AC state! Decompression might fail.")
                   # Use heuristic? If we know how many bits were read?
                   # Not easy. Adding file_pos to save logic in process() is needed.
                   
              print(f"Restored AC state. File position: {inp.tell()}")

          else:
              print("Warning: AC state not found for resume! This will likely fail.")

      process(False, length, vocab_size, dec, output)
      
      # The decompressed data is stored in the "output" list. We can now write the
      # data to file (based on the type of preprocessing used).
      
      # Helper to determine start index for writing
      # We only want to write the *new* part.
      # process() fills `output`.
      # If `current_part > 1`, we loaded `output` partially.
      # We should append only the new symbols.
      
      # Calculate range processed
      split = math.ceil(length / batch_size)
      part_len = math.ceil(split / total_parts)
      start_pos = (current_part - 1) * part_len
      end_pos = min(start_pos + part_len, split)
      
      # We need to map `start_pos` (which is in "steps" or "batches") to actual symbol indices.
      # The `process` loop accesses `index = pos + 1 + i * split`
      # So for each `pos` in `[start_pos, end_pos)`, we write `batch_size` symbols.
      # But the file writing logic in original code iterates `range(length)`.
      # We should efficiently append.

      # Determine which indices in `output` were newly updated?
      # They are scattered because of the `i*split` stride.
      # It's better to rewrite the whole file? 
      # Or just seek and write?
      
      # If we loaded `output` from file, `output` is consistent.
      # We can just write the whole thing if `current_part` overlaps?
      # Wait, `output_path` is "data/decompressed.dat".
      # If we used `ab` (append), we imply contiguous writing.
      # But the `output` array is filled in scattered chunks (strided).
      # If `batch_size > 1`, the file is NOT filled contiguously from start to end in linear order.
      # It is filled: 0, split, 2*split... then 1, split+1...
      
      # This means we CANNOT stream the output to a file sequentially during the process easily if batch_size > 1.
      # The original code writes the *entire* `output` list at the end.
      
      # If we are splitting sessions, we are filling "stripes" of the file.
      # Session 1 fills:
      # [0 ... part_len], [split ... split+part_len], [2*split ... ]
      
      # This means the "partial" file on disk is NOT a prefix of the final file.
      # It is a sparse file or a full file with holes (zeros).
      # If we write it out as a normal file, it will have zeros in the gaps.
      
      # When we load it back:
      # We read the file (which has zeros in gaps).
      # We fill `output`.
      # Then we process Part 2, filling the gaps.
      # Then we write the file again (overwrite).
      
      # So:
      # 1. We should NOT use "append" mode for `output_path`. We should overwrite or update.
      # 2. When loading context, we read the full file (which contains 0s for future parts).
      
      mode_write = "r+b" if (current_part > 1 and os.path.exists(output_path)) else "wb"
      
      with open(output_path, mode_write) as out:
        if preprocess == 'nncp' or preprocess == 'nncp-done':
            for i in range(length):
                # If we are just updating, we could seek... but `i` goes 0..length.
                # Writing everything is safer to ensure consistency, though slow for IO.
                # Optimized: We could only write if we know it changed, but `output` doesn't track that.
                # Given `enwik8` is small, writing 100MB is fast. `enwik9` 1GB is also okay.
                out.write(bytes(((output[i] // 256),)))
                out.write(bytes(((output[i] % 256),)))
        else:
            # Convert indexes back to the original characters.
            idx2char = np.array(vocab)
            for i in range(length):
                 out.write(bytes((idx2char[output[i]],)))

  if preprocess == 'cmix':
    if current_part == total_parts:
        !./cmix/cmix -d ./cmix/dictionary/english.dic $output_path ./data/final.dat
        output_path = "data/final.dat"
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if current_part == total_parts:
        !./nncp-2019-11-16/preprocess d data/dictionary.words $output_path ./data/final.dat
        output_path = "data/final.dat"

In [ ]:
#@title Download Result

if mode == 'decompress':
  if preprocess == 'nncp-done':
    download('data/decompressed.dat')
  else:
    download(output_path)
  if checkpoint:
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

In [ ]:
#@title Validation

if mode == 'decompress' or mode == 'both':
  if preprocess == 'nncp-done':
    !md5sum data/decompressed.dat
  !md5sum $output_path
if mode == 'both':
  !md5sum $original_file

In [ ]:
import time
time.sleep(60)

from google.colab import runtime
runtime.unassign()